In [19]:
!pip install pandas pyarrow langdetect tqdm

In [20]:
import pandas as pd
import re
from langdetect import detect
from tqdm import tqdm
from pathlib import Path

tqdm.pandas()

BASE_DIR = Path.cwd()
CLEANED_DATA_DIR = (BASE_DIR / "../data/cleaned_data").resolve()
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [21]:
df = pd.read_parquet("../data/shared_data/master/comments_master.parquet")  # 文件路径

print("Original size:", len(df))
df.head()

Original size: 133110


,comment_id,video_id,thread_id,parent_comment_id,is_reply,author_channel_id,author_display_name,text_original,text_display,like_count,published_at,updated_at,total_reply_count,search_keyword,fetched_at_utc,fetched_by
0,UgwApYrk9BhCbRRFFzd4AaABAg,31i5MdossTo,UgwApYrk9BhCbRRFFzd4AaABAg,None,False,UCnVP89d_Qu6osTmqbmbaCJg,@stephdobbs-t3f,Cute❤❤❤❤❤❤,Cute❤❤❤❤❤❤,0,2026-04-10 06:03:09+00:00,2026-04-10 06:03:09+00:00,0.0,figurines unboxing,2026-04-10 06:03:24+00:00,member_5
1,UgwY2NHHcdFkEv_cggl4AaABAg,ngc324I-IWM,UgwY2NHHcdFkEv_cggl4AaABAg,None,False,UCzLs8wFctJewJjm9VgzQe1Q,@LatencyLate,Just get a Jason stathom head sculpt and use a...,Just get a Jason stathom head sculpt and use a...,0,2026-04-10 06:02:32+00:00,2026-04-10 06:02:32+00:00,0.0,figurines review,2026-04-10 06:04:11+00:00,member_5
2,UgyChHM8gTyofDJZcYV4AaABAg.AVPJ_Byw7Y7AVPgc55vXiN,HPy42rjmYlM,UgyChHM8gTyofDJZcYV4AaABAg,UgyChHM8gTyofDJZcYV4AaABAg,True,UCHKINgvOgV7FT19ByPhfUdQ,@KingMATTtheSuperior,Blanket rolls and bags would make a nice addit...,Blanket rolls and bags would make a nice addit...,0,2026-04-10 06:02:29+00:00,2026-04-10 06:02:29+00:00,NaN,figurines review,2026-04-10 06:04:07+00:00,member_5
3,Ugx0WD-ByYecy8A8xyx4AaABAg,JiKza7DkQ98,Ugx0WD-ByYecy8A8xyx4AaABAg,None,False,UClMkwbl4Sd1szWtIcW8IMpA,@Dolls.and.Ponies,Giant Barbie😂 her face looks like the Karl fac...,Giant Barbie😂 her face looks like the Karl fac...,0,2026-04-10 06:01:14+00:00,2026-04-10 06:01:14+00:00,0.0,collectibles unboxing,2026-04-10 06:02:17+00:00,member_5
4,UgxEUqssJ5xykb-8T_t4AaABAg,khHRah2q1zc,UgxEUqssJ5xykb-8T_t4AaABAg,None,False,UCJM5Ly2wM89MXDgPsm04uUw,@SubhamPal-iq3xi,Yesyes❤❤❤😊😊😊🎉🎉🎉,Yesyes❤❤❤😊😊😊🎉🎉🎉,1,2026-04-10 06:01:03+00:00,2026-04-10 06:01:03+00:00,0.0,makeup kit unboxing,2026-04-10 06:04:39+00:00,member_5


In [22]:
print(df.columns)

Index(['comment_id', 'video_id', 'thread_id', 'parent_comment_id', 'is_reply',
       'author_channel_id', 'author_display_name', 'text_original',
       'text_display', 'like_count', 'published_at', 'updated_at',
       'total_reply_count', 'search_keyword', 'fetched_at_utc', 'fetched_by'],
      dtype='object')


In [23]:
# 去重
df = df.drop_duplicates(subset=["text_original"])
print("After dedup:", len(df))

After dedup: 122949


In [24]:
# 删除空值 & 太短评论
df = df[df["text_original"].notna()]
df = df[df["text_original"].str.len() > 10]

print("After removing empty/short:", len(df))

After removing empty/short: 115043


In [25]:
# 文本清洗函数
def clean_text(text):
    text = text.lower()
    
    # remove urls
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # remove emoji / non-ascii
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    
    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [26]:
df["clean_text"] = df["text_original"].progress_apply(clean_text)

df[["text_original", "clean_text"]].head()

100%|██████████| 115043/115043 [00:00<00:00, 225704.19it/s]


,text_original,clean_text
1,Just get a Jason stathom head sculpt and use a...,just get a jason stathom head sculpt and use a...
2,Blanket rolls and bags would make a nice addit...,blanket rolls and bags would make a nice addit...
3,Giant Barbie😂 her face looks like the Karl fac...,giant barbie her face looks like the karl face...
4,Yesyes❤❤❤😊😊😊🎉🎉🎉,yesyes
5,He’s got drop down hips,he s got drop down hips


In [27]:
# 语言过滤

def is_english(text):
    try:
        return detect(text) == "en"
    except:
        return False

df = df[df["clean_text"].progress_apply(is_english)]

print("After language filter:", len(df))

100%|██████████| 115043/115043 [02:27<00:00, 780.00it/s]

After language filter: 82584


In [28]:
# 删除低价值评论

LOW_VALUE = [
    "lol", "nice", "wow", "cool", "first", "ok", "okay", "yes", "no"
]

def is_meaningful(text):
    words = text.split()
    
    if len(words) < 3:
        return False
    
    if text in LOW_VALUE:
        return False
        
    return True

df = df[df["clean_text"].progress_apply(is_meaningful)]

print("After removing low-value:", len(df))

100%|██████████| 82584/82584 [00:00<00:00, 1003573.53it/s]

After removing low-value: 81009


In [29]:
# 过滤关键词

KEYWORDS = [
    "case", "cover", "protect", "damage", "drop",
    "scratch", "hard shell", "storage", "carry",
    "bag", "box", "durable", "safe"
]

def contains_keywords(text):
    return any(k in text for k in KEYWORDS)

df = df[df["clean_text"].progress_apply(contains_keywords)]

print("After keyword filter:", len(df))

100%|██████████| 81009/81009 [00:00<00:00, 690094.63it/s]

After keyword filter: 7706


In [30]:
df["clean_text"]

2         blanket rolls and bags would make a nice addit...
12        i was at an ad in michigan at somerset mall in...
17             better a grand seiko what a piece of garbage
55                                          great box coach
101       all the protection you really should be wearin...
                                ...                        
133023    what one smart watch feature do you use the mo...
133053    weird that in the box there s not a lot of cam...
133102    i might wanna add to my gear just a little 4x4...
133105    hi nata this was interesting to see how much t...
133107    this is my b camera travel bag not a perfect s...
Name: clean_text, Length: 7706, dtype: object

In [31]:
df.to_parquet(CLEANED_DATA_DIR / "cleaned_comments.parquet", index=False)

print("✅ Saved as parquet!")

✅ Saved as parquet!


In [32]:
df.to_csv(CLEANED_DATA_DIR / "cleaned_comments.csv", index=False)

print("✅ CSV export done!")

✅ CSV export done!


In [33]:
# print("Final dataset size:", len(df))

# print("\nDemand hint ratio:")
# print(df["is_demand_hint"].value_counts(normalize=True))